In [1]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances, manhattan_distances


c:\Users\ANANDHU\OneDrive\Desktop\Querytube AI\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL_NAME = "all-MiniLM-L6-v2"
METRIC_NAME = "cosine"
TOP_K = 5
THRESHOLD = 0.5
TITLE_WEIGHT = 0.3
TRANSCRIPT_WEIGHT = 0.7

In [3]:
df = pd.read_csv("cleaned_transcripts.csv")

print("Dataset loaded successfully")
print("Shape:", df.shape)

df.head()

Dataset loaded successfully
Shape: (398, 4)


,video_id,title,datetime,transcript
0,E-CH3-VyVck,Why is Git INSANELY Fast? (And How Commits Ac...,2026-02-24 11:00:29+00:00,You have typed get commit thousands of times. ...
1,URI5GsOBznk,YouTube Recommendation Engine: Complete Meltdo...,2026-02-21 04:26:16+00:00,"YouTube went down. [music] 350,000 users repor..."
2,8d2eG7bdepQ,Implementing OAuth and MFA: Full Authenticatio...,2026-02-18 11:00:16+00:00,Every time you click sign in with Google or co...
3,GQ6piqfwr5c,"How Stripe Built AI Agents That Write 1,000+ P...",2026-02-14 13:02:10+00:00,Scribe just revealed something big. They have ...
4,p5hA8rpCRXw,The Selenium Problem: Why QA Teams Waste 40% o...,2026-02-11 11:01:25+00:00,Here's a stat that surprised me. QA team spend...


In [4]:
model = SentenceTransformer(MODEL_NAME)
print("Loaded model:", MODEL_NAME)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4349.36it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded model: all-MiniLM-L6-v2


In [5]:
titles = df["title"].fillna("").astype(str).tolist()
transcripts = df["transcript"].fillna("").astype(str).tolist()

title_embeddings = model.encode(titles, show_progress_bar=True)
transcript_embeddings = model.encode(transcripts, show_progress_bar=True)

print("Title embeddings shape:", title_embeddings.shape)
print("Transcript embeddings shape:", transcript_embeddings.shape)

Batches: 100%|██████████| 13/13 [00:09<00:00,  1.38it/s]

Title embeddings shape: (398, 384)
Transcript embeddings shape: (398, 384)


In [6]:
def compute_scores(query_embedding, metric_name=METRIC_NAME):
    query_embedding = query_embedding.reshape(1, -1)

    if metric_name == "cosine":
        title_scores = cosine_similarity(query_embedding, title_embeddings)[0]
        transcript_scores = cosine_similarity(query_embedding, transcript_embeddings)[0]

    elif metric_name == "euclidean":
        title_scores = -euclidean_distances(query_embedding, title_embeddings)[0]
        transcript_scores = -euclidean_distances(query_embedding, transcript_embeddings)[0]

    elif metric_name == "manhattan":
        title_scores = -manhattan_distances(query_embedding, title_embeddings)[0]
        transcript_scores = -manhattan_distances(query_embedding, transcript_embeddings)[0]

    else:
        raise ValueError("Unsupported metric")

    return title_scores, transcript_scores

In [7]:
def final_search(query, df):
    # 1. Encode query
    query_embedding = model.encode([query])[0]

    # 2. Compute title and transcript scores
    title_scores, transcript_scores = compute_scores(query_embedding, METRIC_NAME)

    # 3. Combine scores
    final_scores = (TITLE_WEIGHT * title_scores) + (TRANSCRIPT_WEIGHT * transcript_scores)

    # 4. Copy dataframe and attach scores
    results_df = df.copy()
    results_df["title_score"] = title_scores
    results_df["transcript_score"] = transcript_scores
    results_df["final_score"] = final_scores

    # 5. Apply threshold filtering
    filtered_df = results_df[results_df["final_score"] >= THRESHOLD]

    # 6. Sort and keep top-k
    filtered_df = filtered_df.sort_values(by="final_score", ascending=False).head(TOP_K)

    # 7. Create YouTube links
    filtered_df["youtube_link"] = filtered_df["video_id"].apply(
        lambda x: f"https://www.youtube.com/watch?v={x}"
    )

    return filtered_df[["title", "video_id", "final_score", "youtube_link"]]

In [12]:
user_query = input("Enter your search query: ")

results = final_search(user_query, df)

print(f"\nTop {TOP_K} relevant videos for query: {user_query}\n")

for i, row in results.iterrows():
    print(f"Title: {row['title']}")
    print(f"Video ID: {row['video_id']}")
    print(f"Score: {row['final_score']:.3f}")
    print(f"Link: {row['youtube_link']}")
    print("-" * 60)


Top 5 relevant videos for query: what is quantum computer?

Title: Quantum Computing Explained Simply | How Qubits Power the Future
Video ID: LM3rwbVPyNc
Score: 0.623
Link: https://www.youtube.com/watch?v=LM3rwbVPyNc
------------------------------------------------------------
